<a href="https://colab.research.google.com/github/DM871/dm871.github.io/blob/master/notebooks/lab_highspy_2.ipynb">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>

# Solving MILP Problems in Python - Part 2

## The Diet Example

So far we have written models with embedded data. However, when building
an optimization model, it is typical to separate the optimization model
itself from the data used to create an instance of the model. These two
model ingredients are often stored in completely different files.

There are alternate approaches to providing data to the optimization
model: they can be embedded in the source file, read from an SQL
database (using the Python sqlite3 package), or read them from an Excel
spreadsheet (using the Python xlrd package) and more.

#### Diet Problem

Bob wants to plan a nutritious diet, but he is on a limited budget, so
he wants to spend as little money as possible. His nutritional
requirements are as follows:
```
  ----------------
  2000 Kcal
  55 g protein
  800 mg calcium
  ----------------
```
Bob is considering the following foods with corresponding nutritional
values
```
                Serving Size   Price per serving   Energy (Kcal)   Protein (g)   Calcium (mg)
  ----------- -------------- ------------------- --------------- ------------- --------------
  Oatmeal               28 g                 0.3             110             4              2
  Chicken              100 g                 2.4             205            32             12
  Eggs               2 large                 1.3             160            13             54
  Milk                237 cc                 0.9             160             8            285
  Apple Pie            170 g                   2             420             4             22
  Pork                 260 g                 1.9             260            14             80
```
With the help of HiGHS Python (`highspy`), find the amount of servings of each type
of food in the diet.

We specify the model independently from the data. We could put the model in a file, eg, `dietmodel.py` and the data in another file, eg, `diet1.py`. Make
sure you understand the model and read about the method
`Highs.qsum` in the [Python interface](https://ergo-code.github.io/HiGHS/dev/interfaces/python/)
documentation, which efficiently sums up a sequence of linear expressions.


In [ ]:
%pip install highspy

In [ ]:
import highspy

In [ ]:
def solve(categories, minNutrition, maxNutrition, foods, cost, nutritionValues):
    # Model
    m = highspy.Highs()

    # Create decision variables for the nutrition information,
    # which we limit via bounds
    nutrition = {}
    for c in categories:
        nutrition[c] = m.addVariable(lb=minNutrition[c], ub=maxNutrition[c], name=c)

    # Create decision variables for the foods to buy
    buy = {}
    for f in foods:
        buy[f] = m.addVariable(obj=cost[f], name=f.replace(" ", "_"))  # LP format does not allow spaces in names

    # The objective is to minimize the costs
    m.changeObjectiveSense(highspy.ObjSense.kMinimize)

    # Nutrition constraints
    for c in categories:
        m.addConstr(m.qsum(nutritionValues[f, c] * buy[f] for f in foods) == nutrition[c], name=c)

    def printSolution():
        if m.getModelStatus() == highspy.HighsModelStatus.kOptimal:
            print('\nCost: %g' % m.getObjectiveValue())
            print('\nBuy:')
            for f in foods:
                if m.val(buy[f]) > 0.0001:
                    print('%s %g' % (f, m.val(buy[f])))
            print('\nNutrition:')
            for c in categories:
                print('%s %g' % (c, m.val(nutrition[c])))
        else:
            print('No solution')

    # Solve
    m.writeModel("diet.lp")
    print(open("diet.lp").read())
    m.run()
    printSolution()

To arrange the data in Python data structures we use the following
function `multidict` (it mimics the homonymous function of `gurobipy`). It takes a
dictionary whose values are lists and splits it into a list of keys and
one dictionary for each position in the lists. If the values are scalars,
it returns the list of keys and a single dictionary. Here is an example of what it
does:

In [ ]:
def multidict(data):
    keys = list(data.keys())
    values = list(data.values())
    if not isinstance(values[0], (list, tuple)):
        return keys, dict(data)
    return [keys] + [{k: v[i] for k, v in data.items()} for i in range(len(values[0]))]

keys, dict1, dict2 = multidict( {
        'key1': [1, 2],
        'key2': [1, 3],
        'key3': [1, 4] } )
print(keys, dict1, dict2)

In [ ]:
categories, minNutrition, maxNutrition = multidict({
  'Calories': [1800, 2200],
  'Protein':  [91, highspy.kHighsInf],
  'Calcium':   [0, 1779] })

foods, cost = multidict({
	'Oatmeal':           	 0.30,
	'Chicken':		 2.40,
	'Eggs':   	         1.30,
	'Milk':             	 0.90,
	'Apple Pie':           	 2.00,
	'Pork':              	 1.90});

# Nutrition values for the foods
nutritionValues = {
('Oatmeal',           'Calories'  ):	110, 
('Oatmeal',           'Protein' ):	4,    
('Oatmeal',           'Calcium' ):	2,
('Chicken',           'Calories'  ):	205, 
('Chicken',           'Protein' ):	32,    
('Chicken',          'Calcium' ):	12, 
('Eggs',              'Calories'  ):	160,       
('Eggs',              'Protein' ):	13,   
('Eggs',              'Calcium' ):	54, 
('Milk',              'Calories'  ):	160,       
('Milk',             'Protein' ):	8,            
('Milk',              'Calcium' ):	285,   
('Apple Pie',        'Calories'  ):	420, 
('Apple Pie',        'Protein' ):	4,            
('Apple Pie',        'Calcium' ):	22, 
('Pork',              'Calories'  ):	260, 
('Pork',              'Protein' ):	14,           
('Pork',              'Calcium' ):	80 };


If we put the code in the two files, then we could solve from `diet1.py` as follows:
```python
import dietmodel
dietmodel.solve(categories, minNutrition, maxNutrition, foods, cost, nutritionValues)
```

Here it suffices calling:

In [ ]:
solve(categories, minNutrition, maxNutrition, foods, cost, nutritionValues)

#### Your Task

A pill salesman offers Bob Calories, Protein, and Calcium pills to
fulfill his nutritional needs. He needs to estimate the prices of units
of serving, that is, the cost of 1 kcal, the cost of 1 g of protein, the
cost of 1 mg of calcium. He wants to make as much money as possible,
given Bob's constraints. He knows that Bob wants 2200 kcal, 55 g
protein, and 1779 mg calcium. How can we help him in guaranteeing that
he does not make a bad deal?



**Solution:**

The dual seeks to maximize the profit of the salesman. Let $y_i \geq 0$,
$i\in N$ be the prices of the pills.

$$\begin{aligned}
\text{min}\quad &\sum_{j \in F}c_jx_j\\
\sum_{j\in F} a_{ij}x_j&\geq N_{min,i}, \qquad\forall i \in N\\
%\sum_{j\in F}a_{ij}x_{j}&\leq N_{max,i}, \qquad\forall i \in N\\
x_j&\geq 0, \qquad \forall j\in F\\
x_j&\leq F_{max,j}, \qquad \forall j \in F\end{aligned}$$

$$\begin{aligned}
\text{max}\quad &\sum_{i \in F}N_{min,i}y_i\\
\sum_{i\in N} a_{ji}y_i&\leq c_{j}, \qquad\forall j \in F\\
y_i&\geq 0, \qquad \forall i\in N\\\end{aligned}$$

However the values of the dual variables can be determined by the last
tableau of the solution to the primal problem by printing the dual values
of the constraints, that is, `m.constrDual(c)` or the list `m.getSolution().row_dual`.


# Particular Cases

The two following LP problems lead to two particular cases when solved
by the simplex algorithm. Identify these cases and characterize them,
that is, give indication of which conditions generate them in general.
Then, implement the models in HiGHS Python and observe the behaviour.

$$\begin{array}{rllllllllll}
\mbox{maximize} & 2x_1& +& x_2&\\
    \mbox{subject to} &&&   x_2&  \leq &5&\\
&                    - x_1& +& x_2 &\leq &1& \\
&              x_1, &&x_2&\geq &0& \\
\end{array}$$

$$\begin{array}{rlllllllllll}
\mbox{maximize}& x_1 &+& x_2&\\
    \mbox{subject to} & 5 x_1& +& 10 x_2&  \leq& 60&\\
&                    4 x_1 &+ &4 x_2 &\leq &40& \\
&               x_1, &&x_2&\geq &0& \\
\end{array}$$

# Pathological Cases

This exercise asks you to check the behavior of the solvers on the two
pathological cases:

$$\begin{array}{rlllllllllll}
\mbox{maximize}&&& 4x_2&\\
\mbox{subject to}&&& 2x_2 &\geq& 0\\
& -3x_1& +&4 x_2& \geq &1\\
&       x_1, &&x_2&\geq &0& \\
\end{array}$$

$$\begin{array}{rlllllllllll}
\mbox{maximize}\ \ \   10x_1 -57 x_2 -9x_3-24x_4&\\
    \mbox{subject to}\ \   -0.5x_1+5.5x_2+2.5x_3-9x_4 &\leq 0\\
                                  -0.5 x_1 + 1.5 x_2 +0.5 x_3 -x_4& \leq 0\\
  x_1  &\leq 1&\\
               x_1, x_2, x_3, x_4&\geq 0& \\
\end{array}$$

What happens with the solver? Can you detect which pathological cases
are from the output of the solver? How?


# Shortest Path

Model the shortest path problem as an LP problem. Write the model in
Python using the skeleton below that reads the coordinates of 20 data points.
In this data the source is node 1
and the target is node 20 and we assume that all connections between vertices are available except the direct link from source to target.

Model the problem in LP and solve it with HiGHS Python. Check the
correctness of your solution with the help of the visualization in the
template below.

For the implementation it may be helpful to use the method `Highs.addVariables`,
which creates one variable for each element of a list of keys (here the arcs)
and returns a dictionary from keys to variables, and list comprehensions
to select the arcs entering and leaving a vertex.

Spoiler: the code below contains the generation of the instance and the answer to your task. Try first to address the task by your own and then check the solution.


In [ ]:
import math
import pandas as pd
import highspy


url = 'https://dm871.github.io/assets/20points.csv'
df = pd.read_csv(url, sep='\t', header=None, names=('label','x','y'))
points = {v: (x,y) for v,x,y in df.itertuples(index=False)}


source = 1
target = 20

# calculate Euclidean distance and round-towards-zero (truncate)

def distance(points, i, j):
    dx = points[i][0] - points[j][0]
    dy = points[i][1] - points[j][1]
    return math.floor(math.sqrt(dx*dx + dy*dy))

m = highspy.Highs()

V = points.keys()
A = [(i,j) for i in V for j in V if i!=j]
A.remove((source,target))

In [ ]:
##############################################################################
# Begin: Answer to your task
##############################################################################

# Create variables
x = m.addVariables(A, obj=[distance(points,i,j) for i,j in A], name_prefix='x')

# Post the constraints
def balance(v):
    if v==source:
        return 1
    if v==target:
        return -1
    return 0

for v in V:
    m.addConstr(m.qsum(x[v,u] for (w,u) in A if w==v) - m.qsum(x[u,v] for (u,w) in A if w==v) == balance(v), name="balance_%d" % v)

##############################################################################
# End: Answer to your task
##############################################################################


m.writeModel("sp.lp");

In [ ]:
m.run()
solution = m.vals(x)

selected = [(i,j) for i,j in A if solution[i,j] > 0.5]
print(list(filter(lambda x: x[1]>0, solution.items())))

print('Optimal path: %s' % str(selected))
print('Optimal cost: %g' % m.getObjectiveValue())

In [ ]:
%matplotlib inline

In [ ]:

import matplotlib.pyplot as plt
def plot_path(points, path, style='bo-'):
    "Plot lines to connect a series of points."
    plt.plot([p[1][0] for p in points.items()], [p[1][1] for p in points.items()], 'bo')
    plt.plot([points[source][0],points[target][0]],[points[source][1],points[target][1]], 'rs')
    plt.plot([(points[p[0]][0],points[p[1]][0]) for p in path], 
             [(points[p[0]][1],points[p[1]][1]) for p in path], 'b-')
    plt.axis('scaled'); plt.axis('off')
    plt.show()

print(points, selected)
plot_path(points, selected)